In [ ]:
# Import necessary libraries and initialize GEE
import ee
import ee.mapclient

# Initialize Earth Engine
# ee.Authenticate()
ee.Initialize(project='qsair-463811')

In [ ]:
# Define an agricultural area of interest (AOI)
# This example defines a simple rectangle. In a real scenario, you might
# import a shapefile or define a more complex geometry.
# Coordinates are typically [min_longitude, min_latitude, max_longitude, max_latitude]
agricultural_area = ee.Geometry.Rectangle([-99.65, 39.97, -99.64, 39.98])

In [ ]:
# Load Sentinel-2 imagery and filter
# Filter by date, cloud percentage, and bounds
# You might need to adjust the date range based on your study area and desired season.
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2022-07-01', '2022-07-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .filterBounds(agricultural_area)

# Select the median image to get a representative scene for the period
median_image = collection.median()

In [ ]:
# Calculate various spectral indices
# Normalizing Difference (ND) indices are common in remote sensing for various features.
# The general formula is (Band1 - Band2) / (Band1 + Band2)

# Normalized Difference Red Edge Index (NDRE)
# More sensitive to chlorophyll content than NDVI
# (NIR - RED_EDGE) / (NIR + RED_EDGE)
ndre = median_image.normalizedDifference(['B8', 'B5']).rename('NDRE')

# Modified Chlorophyll Absorption Ratio Index (MCARI)
# More sensitive to chlorophyll content than NDVI, especially in dense canopies.
# MCARI = ((RED_EDGE - RED) - 0.2 * (RED_EDGE - GREEN)) * (RED_EDGE / RED)
mcari = median_image.expression(
    '((B5 - B4) - 0.2 * (B5 - B3)) * (B5 / B4)',
    {
        'B3': median_image.select('B3'), # Green
        'B4': median_image.select('B4'), # Red
        'B5': median_image.select('B5')  # Red Edge
    }).rename('MCARI')

# Plant Senescence Reflectance Index (PSRI)
# Tracks plant senescence and stress
# PSRI = (RED - BLUE) / RED_EDGE
psri = median_image.expression(
    '(B4 - B2) / B5',
    {
        'B2': median_image.select('B2'), # Blue
        'B4': median_image.select('B4'), # Red
        'B5': median_image.select('B5')  # Red Edge
    }).rename('PSRI')

# Visible Atmospherically Resistant Index (VARI)
# Designed to be resistant to atmospheric effects for estimating vegetation fraction.
# VARI = (GREEN - RED) / (GREEN + RED - BLUE)
vari = median_image.expression(
    '(B3 - B4) / (B3 + B4 - B2)',
    {
        'B2': median_image.select('B2'), # Blue
        'B3': median_image.select('B3'), # Green
        'B4': median_image.select('B4')  # Red
    }).rename('VARI')

# Normalized Difference Vegetation Index (NDVI)
# Most common vegetation index, (NIR - RED) / (NIR + RED)
ndvi = median_image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Enhanced Vegetation Index (EVI)
# Improves on NDVI by decoupling the canopy background signal and reducing atmospheric influences.
# EVI = 2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))
evi = median_image.expression(
    '2.5 * ((B8 - B4) / (B8 + 6 * B4 - 7.5 * B2 + 1))',
    {
        'B2': median_image.select('B2'), # Blue
        'B4': median_image.select('B4'), # Red
        'B8': median_image.select('B8')  # NIR
    }).rename('EVI')

# Soil Adjusted Vegetation Index (SAVI)
# Similar to NDVI but accounts for soil brightness variations.
# SAVI = ((NIR - RED) / (NIR + RED + L)) * (1 + L) where L is a soil brightness correction factor (usually 0.5)
savi = median_image.expression(
    '(1 + 0.5) * ((B8 - B4) / (B8 + B4 + 0.5))',
    {
        'B4': median_image.select('B4'), # Red
        'B8': median_image.select('B8')  # NIR
    }).rename('SAVI')

# Merge all calculated indices into a single image for easier handling
all_indices = ndre.addBands(mcari).addBands(psri).addBands(vari).addBands(ndvi).addBands(evi).addBands(savi)

In [ ]:
# Define visualization parameters for each index and add layers to the map
# Each index needs a color palette and min/max values for proper visualization.
# These values are typical ranges, but might need adjustment based on your specific data.

# Define visualization parameters for each index
ndre_vis = {'min': -0.1, 'max': 0.8, 'palette': ['red', 'yellow', 'green']}
mcari_vis = {'min': 0, 'max': 0.5, 'palette': ['red', 'yellow', 'green']}
psri_vis = {'min': -0.2, 'max': 0.2, 'palette': ['green', 'yellow', 'red']}
vari_vis = {'min': 0, 'max': 0.75, 'palette': ['red', 'yellow', 'green']}
ndvi_vis = {'min': -0.2, 'max': 0.9, 'palette': ['red', 'yellow', 'green']}
evi_vis = {'min': 0, 'max': 1, 'palette': ['red', 'yellow', 'green']}
savi_vis = {'min': 0, 'max': 1, 'palette': ['red', 'yellow', 'green']}

# Add layers to map (with only 1-2 visible at a time)
ee.mapclient.addToMap(ndvi, ndvi_vis, 'NDVI', shown=True)
ee.mapclient.addToMap(evi, evi_vis, 'EVI', shown=False)
ee.mapclient.addToMap(ndre, ndre_vis, 'NDRE', shown=False)
ee.mapclient.addToMap(mcari, mcari_vis, 'MCARI', shown=False)
ee.mapclient.addToMap(psri, psri_vis, 'PSRI', shown=False)
ee.mapclient.addToMap(vari, vari_vis, 'VARI', shown=False)
ee.mapclient.addToMap(savi, savi_vis, 'SAVI', shown=False)

In [ ]:
# Export results (optional, for further analysis outside GEE)
# This example shows how to export a specific band (e.g., NDVI) to Google Drive.
# You can modify this to export other bands or the entire 'all_indices' image.

# Export NDVI to Drive
# ee.batch.Export.image.toDrive(
#     image=ndvi,
#     description='NDVI_Export',
#     folder='GEE_Exports', # Specify a folder in your Google Drive
#     scale=10, # Resolution in meters per pixel
#     region=agricultural_area.getInfo() # Export within the defined AOI
# ).start()

# Export all indices as a multi-band image (example)
# ee.batch.Export.image.toDrive(
#     image=all_indices,
#     description='All_Indices_Export',
#     folder='GEE_Exports',
#     scale=10,
#     region=agricultural_area.getInfo()
# ).start()

In [ ]:
# Create a custom point for inspecting values
# This allows you to click on the map and retrieve index values at that specific location.
point_of_interest = ee.Geometry.Point([-99.6, 39.95]) # Example coordinates

# Add the point to the map
ee.mapclient.addToMap(point_of_interest, {'color': 'blue'}, 'Sample Point')

# Get values at the point for the median image and all calculated indices
median_image_values = median_image.reduceRegion(ee.Reducer.mean(), point_of_interest, 10).getInfo()
all_indices_values = all_indices.reduceRegion(ee.Reducer.mean(), point_of_interest, 10).getInfo()

print("Median Image Values at Sample Point:", median_image_values)
print("Advanced Spectral Index Values at Sample Point:", all_indices_values)

In [ ]:
# Compare indices for a specific key (e.g., 'NDVI')
# This is useful if you want to quickly see the value of a particular index at the point.
selected_index_key = 'NDVI'
if selected_index_key in all_indices_values:
    print(f"Value of {selected_index_key} at sample point: {all_indices_values[selected_index_key]}")
else:
    print(f"{selected_index_key} not found in the calculated indices.")

In [ ]:
# Form boundary around sample point (optional)
# This might be used for creating small buffer zones or sampling areas around points.
# For example, to get statistics within a small radius around the point.
# Here, it creates a small circle (buffer) around the point.
form_boundary_around_point = point_of_interest.buffer(100) # 100 meters buffer

# Add the boundary to the map for visualization
ee.mapclient.addToMap(form_boundary_around_point, {'color': 'orange'}, 'Point Buffer (100m)')

In [ ]:
# Further analysis within the formed boundary (example: zonal statistics)
# This cell demonstrates how you might extract data for further analysis within the buffer.
# For example, calculating the mean of all indices within the 100m buffer.
zonal_stats = all_indices.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=form_boundary_around_point,
    scale=10 # Use the same scale as your imagery
).getInfo()

print("Mean of Advanced Spectral Indices within 100m buffer:", zonal_stats)